# Day 8 — Aggregations & GroupBy

**What you will learn:**
- `groupBy` + `agg` with multiple functions in one call
- All core aggregate functions
- Filtering after groupBy (SQL `HAVING` equivalent)
- `pivot` — turning row values into columns
- `describe()` for quick stats

**Exam relevance:** DataFrame API (30%) — aggregations appear in almost every exam question set.

In [ ]:
from pyspark.sql.functions import col, sum, avg, count, min, max, countDistinct, round

data = [
    ("Alice",   "Engineering", "RO", 95000, 2022),
    ("Bob",     "Marketing",   "RO", 72000, 2021),
    ("Carol",   "Engineering", "UK", 110000, 2020),
    ("Dave",    "Sales",       "DE", 58000,  2023),
    ("Eve",     "Marketing",   "RO", 81000,  2021),
    ("Frank",   "Engineering", "DE", 88000,  2022),
    ("Grace",   "Sales",       "RO", 61000,  2020),
    ("Hank",    "Engineering", "UK", 102000, 2023),
    ("Iris",    "Marketing",   "DE", 76000,  2022),
    ("Jack",    "Sales",       "UK", 67000,  2021),
]
df = spark.createDataFrame(data, ["name", "dept", "country", "salary", "hire_year"])
df.show()

## 1. groupBy + agg — Multiple Functions in One Call

> **Exam tip:** Always use `.agg()` with named imports from `pyspark.sql.functions` — not string shorthand — so you can alias results cleanly.

In [ ]:
# Multiple aggregations in a single .agg() call
df.groupBy("dept").agg(
    count("*").alias("headcount"),
    round(avg("salary"), 0).alias("avg_salary"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary"),
    sum("salary").alias("total_salary"),
    countDistinct("country").alias("countries"),
).orderBy(col("avg_salary").desc()).show()

## 2. Filter After groupBy — HAVING Equivalent

In SQL: `GROUP BY dept HAVING headcount > 2`  
In Spark: chain `.filter()` after `.agg()`

In [ ]:
# HAVING equivalent — filter on aggregated column
df.groupBy("dept") \
  .agg(count("*").alias("headcount"), avg("salary").alias("avg_salary")) \
  .filter(col("headcount") > 2) \
  .orderBy("dept") \
  .show()

## 3. GroupBy Multiple Columns

In [ ]:
# Group by dept AND country
df.groupBy("dept", "country") \
  .agg(count("*").alias("n"), avg("salary").alias("avg_sal")) \
  .orderBy("dept", "country") \
  .show()

## 4. pivot — Row Values Become Columns

`pivot` is a wide transformation — it requires a shuffle and is expensive on high-cardinality columns.

> **Exam tip:** Providing the list of pivot values explicitly is faster — Spark doesn't need a pre-scan to find distinct values.

In [ ]:
# Pivot: dept rows → country columns, values = avg salary
df.groupBy("dept") \
  .pivot("country", ["RO", "UK", "DE"]) \
  .agg(round(avg("salary"), 0)) \
  .show()

# Pivot hire_year into columns — headcount per dept per year
df.groupBy("dept") \
  .pivot("hire_year", [2020, 2021, 2022, 2023]) \
  .agg(count("*")) \
  .fillna(0) \
  .show()

## 5. describe() — Quick Summary Stats

In [ ]:
# describe() runs count, mean, stddev, min, max for numeric columns
df.describe("salary", "hire_year").show()

# summary() adds percentiles
df.select("salary").summary().show()

## 6. agg() Without groupBy — Whole-DataFrame Aggregation

In [ ]:
# Aggregate the whole DataFrame (no groupBy)
df.agg(
    count("*").alias("total_rows"),
    round(avg("salary"), 0).alias("company_avg_salary"),
    max("salary").alias("highest_salary"),
).show()

---

## Day 8 Checklist

- [ ] Used `.agg()` with 3+ aggregate functions in one call
- [ ] Aliased every aggregated column
- [ ] Applied HAVING-style filter after `.agg()`
- [ ] Used `pivot()` with explicit value list
- [ ] Used `describe()` and `summary()`
- [ ] Know which aggregation functions are in `pyspark.sql.functions`

**Next:** Day 9 — Joins (all 7 types + broadcast)

## Common Aggregate Functions Reference

| Function | Import | Description |
|---|---|---|
| `count("*")` | built-in | Row count including nulls |
| `count(col)` | built-in | Non-null count |
| `countDistinct(col)` | functions | Distinct non-null values |
| `sum(col)` | functions | Sum |
| `avg(col)` | functions | Mean |
| `min(col)` | functions | Minimum |
| `max(col)` | functions | Maximum |
| `stddev(col)` | functions | Sample std deviation |
| `variance(col)` | functions | Sample variance |
| `first(col)` | functions | First value in group |
| `last(col)` | functions | Last value in group |
| `collect_list(col)` | functions | All values as array |
| `collect_set(col)` | functions | Distinct values as array |